# Evidently

[Evidently](https://www.evidentlyai.com/) is an open-source Python package for evaluating, testing and monitoring machine learning systems. In this notebook we will compare a reference split against later data, inspect the resulting report, and save a reusable reference dataset for the batch monitoring chapter.


In [ ]:
import os
import warnings
from pathlib import Path
from typing import Any, cast
from urllib.request import urlretrieve

import mlflow
import pandas as pd
from dotenv import load_dotenv
from evidently.future.datasets import DataDefinition, Dataset
from evidently.future.metrics import DriftedColumnsCount, MissingValueCount, ValueDrift
from evidently.future.report import Report
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore", category=FutureWarning)

Set the environment variables by copying `.env.example` to `.env`. In a local-first setup, `MLFLOW_TRACKING_URI=./mlruns` is enough to load the model you registered in notebook 01.


In [ ]:
# Load environment variables from .env file
load_dotenv()

# Get the MLflow tracking URI from environment variables
MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "./mlruns")

# Set up the connection to MLflow
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment("green-taxi-monitoring")

First we will load the model from the MLflow model registry. We reuse the `production` alias so the later notebooks and Prefect flow can all point to the same registered model name.


In [ ]:
def load_model(model_name):
    alias = "production"
    model_uri = f"models:/{model_name}@{alias}"
    model = mlflow.pyfunc.load_model(model_uri)
    return model

In [ ]:
model_name = "green-taxi-ride-duration"

loaded_model = load_model(model_name)

Then we prepare the data as we did in the notebook 01:

In [ ]:
# Load the data
year = 2024
month = 1
color = "green"

df = pd.read_parquet(f"./data/{color}_tripdata_{year}-{month:02d}.parquet")

In [ ]:
cat_features = ["PULocationID", "DOLocationID"]
num_features = ["trip_distance", "passenger_count", "fare_amount", "total_amount"]
target = "duration"

In [ ]:
# Calculate duration and keep trips in the same 1 to 60 minute range used earlier.
def calculate_trip_duration_in_minutes(df):
    df["duration"] = (
        df["lpep_dropoff_datetime"] - df["lpep_pickup_datetime"]
    ).dt.total_seconds() / 60
    df = df[(df["duration"] >= 1) & (df["duration"] <= 60)]
    df = df[(df["passenger_count"] > 0) & (df["passenger_count"] < 8)]
    df = df[cat_features + num_features + [target]]
    return df

In [ ]:
df_processed = calculate_trip_duration_in_minutes(df)

In [ ]:
y = df_processed["duration"]
X = df_processed.drop(columns=["duration"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, random_state=42, test_size=0.2
)

In [ ]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

We will make the predictions on the train set first:

In [ ]:
y_pred_train = loaded_model.predict(X_train)

train_data = X_train.copy()
train_data["duration"] = y_train
train_data["prediction"] = y_pred_train

Now on the test set:

In [ ]:
y_pred_test = loaded_model.predict(X_test)

test_data = X_test.copy()
test_data["duration"] = y_test
test_data["prediction"] = y_pred_test

And calculate the mean absolute error (MAE):

In [ ]:
print(mean_absolute_error(train_data.duration, train_data.prediction))
print(mean_absolute_error(test_data.duration, test_data.prediction))

Finally, we will create the **Evidently report**. The first step is to prepare the data for Evidently:

In [ ]:
# Prepare Evidently datasets
val_data = test_data.copy()
data_definition = DataDefinition(
    numerical_columns=num_features + ["prediction"], categorical_columns=cat_features
)


train_dataset = Dataset.from_pandas(train_data, data_definition)

val_dataset = Dataset.from_pandas(val_data, data_definition)

Then we can create the report and include `ValueDrift`, `DriftedColumnsCount` and `MissingValueCount` to detect model drift and data drift. We set `include_tests=True` so the report also includes threshold-based checks.

In [ ]:
report_metrics: list[Any] = [
    cast(Any, ValueDrift)(column="prediction"),
    cast(Any, DriftedColumnsCount)(),
    cast(Any, MissingValueCount)(column="prediction"),
]

report = Report(
    metrics=report_metrics,
    include_tests=True,
)

Now we have to run the report with our data. We will use the train data for the reference and the test data for the production:

In [ ]:
snapshot = report.run(reference_data=train_dataset, current_data=val_dataset)

In [ ]:
snapshot

Or we can save it as a dictionary and get the metrics:

In [ ]:
result = snapshot.dict()
result

In [ ]:
# Prediction drift
result["metrics"][0]["value"]

In [ ]:
# Number of drifted columns
result["metrics"][1]["value"]["count"]

In [ ]:
# Share of missing values
result["metrics"][2]["value"]["share"]

We do not see much drift between the train and test split. That is a good sign: the reference data and the holdout data still look similar enough that the report stays quiet.

But monitoring becomes more useful when we compare that reference against a later batch.


In [ ]:
# Load the data
year = 2025
month = 2
color = "green"

# Download the data
data_path = Path(f"./data/{color}_tripdata_{year}-{month:02d}.parquet")
data_path.parent.mkdir(exist_ok=True)

if not data_path.exists():
    urlretrieve(
        f"https://d37ci6vzurychx.cloudfront.net/trip-data/{color}_tripdata_{year}-{month:02d}.parquet",
        data_path,
    )

df_25_2 = pd.read_parquet(data_path)

Preprocess the data:

In [ ]:
df_processed_25_2 = calculate_trip_duration_in_minutes(df_25_2)

Predict on the new data:

In [ ]:
y_pred_25_2 = loaded_model.predict(df_processed_25_2[cat_features + num_features])

In [ ]:
df_processed_25_2["prediction"] = y_pred_25_2

In [ ]:
# Prepare Evidently current dataset
current_dataset = Dataset.from_pandas(df_processed_25_2, data_definition)

And create the report again:

In [ ]:
snapshot = report.run(reference_data=train_dataset, current_data=current_dataset)

In [ ]:
snapshot

In [ ]:
result = snapshot.dict()
result

Here the report becomes more interesting: even if the prediction drift stays limited, the feature distributions move more noticeably. That is exactly the kind of shift we want to surface before model quality degrades too far.


The final step is to save the training-side data with predictions as a reusable reference snapshot. The Prefect flow from notebook 04 will load this file and compare it to each new batch automatically.


In [ ]:
train_data.to_parquet("./data/reference_data.parquet")